In [1]:
import json
import operator
from pathlib import Path
from typing import TypedDict, Annotated, List, Dict, Any

from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END

# --- CARICAMENTO DATI DAI FILE JSON ALLEGATI ---
DATA_DIR = Path("data")

with open(DATA_DIR / "claims.json", "r", encoding="utf-8") as f:
    CLAIMS_DB = {c["claim_id"]: c for c in json.load(f)} #

with open(DATA_DIR / "customers.json", "r", encoding="utf-8") as f:
    CUSTOMERS_DB = {c["customer_id"]: c for c in json.load(f)} #

with open(DATA_DIR / "policies.json", "r", encoding="utf-8") as f:
    POLICIES_DB = {p["policy_id"]: p for p in json.load(f)} #

with open(DATA_DIR / "requirements.json", "r", encoding="utf-8") as f:
    REQUIREMENTS_DB = {r["loss_type"]: r for r in json.load(f)} #

llm = ChatOllama(model="llama3.2", temperature=0)


# ==============================================================================
# 2. DEFINIZIONE DEI TOOL PER LE 6 TASK
# ==============================================================================
@tool
def verifica_copertura_e_tipo_polizza(policy_id: str, loss_type: str) -> str:
    """Task 3 e Task 4: Verifica se la polizza è attiva, se copre il tipo di evento (Task 3)

    e restituisce il tipo di assicurazione e prodotto (Task 4).
    """
    policy = POLICIES_DB.get(policy_id) #
    if not policy:
        return f"RIFIUTATO — Polizza '{policy_id}' non trovata a sistema." #
    
    if not policy.get("active", False): #
        return f"RIFIUTATO — La polizza '{policy_id}' risulta DISATTIVA." #
        
    coverage = policy.get("coverage", "") #
    if coverage != loss_type: #
        return f"RIFIUTATO — Il tipo evento '{loss_type}' non è coperto dal tipo garanzia '{coverage}'." #
        
    prodotto = policy.get("product", "Sconosciuto") #
    return f"OK — Polizza attiva. Prodotto: '{prodotto}', Copertura: '{coverage}'." #


@tool
def analizza_documenti(claim_documents: List[str], loss_type: str) -> str:
    """Task 1 e Task 6: Ispeziona i documenti forniti dal cliente (Task 6)

    e verifica se tutti i documenti richiesti sono presenti (Task 1).
    """
    req = REQUIREMENTS_DB.get(loss_type) #
    if not req:
        return f"ERRORE — Tipo evento '{loss_type}' non presente nei requisiti." #
        
    req_docs = set(req.get("required_documents", [])) #
    provided_docs = set(claim_documents) #
    
    mancanti = list(req_docs - provided_docs) #
    
    if mancanti:
        return f"INCOMPLETO — Documenti forniti: {list(provided_docs)}. Documenti mancanti: {mancanti}." #
    
    return f"OK — Documentazione completa. Documenti forniti: {list(provided_docs)}." #


@tool
def calcola_budget_e_limiti(estimated_damage: float, policy_id: str, loss_type: str) -> str:
    """Task 2 e Task 5: Verifica franchigia (tetto minimo) e massimale (tetto massimo) (Task 5),

    e controlla il rientro nel budget di approvazione automatica (Task 2).
    """
    policy = POLICIES_DB.get(policy_id) #
    req = REQUIREMENTS_DB.get(loss_type) #
    
    if not policy or not req:
        return "ERRORE — Dati polizza o requisiti mancanti." #
        
    deductible = policy.get("deductible", 0) #
    max_claim = policy.get("max_claim", 0) #
    auto_max = req.get("automatic_review_max", 0) #
    
    # Check 1: Sotto la franchigia (tetto minimo)
    if estimated_damage <= deductible: #
        return f"RIFIUTATO — Danno stimato ({estimated_damage}€) inferiore o pari alla franchigia/tetto minimo ({deductible}€)." #
        
    # Check 2: Supera il massimale (tetto massimo)
    if estimated_damage > max_claim: #
        return f"RIFIUTATO — Danno stimato ({estimated_damage}€) supera il massimale/tetto massimo della polizza ({max_claim}€)." #
        
    # Check 3: Rientro nel budget per approvazione automatica
    if estimated_damage > auto_max: #
        return f"ESCALATION — Danno stimato ({estimated_damage}€) supera la soglia di approvazione automatica ({auto_max}€). Richiesta revisione manuale." #
        
    return f"OK — Danno stimato ({estimated_damage}€) compreso tra franchigia ({deductible}€) e massimale ({max_claim}€), ed entro la soglia automatica ({auto_max}€)." #

